# LLM Fine-Tuning Data Preparation

This notebook prepares context-aware supervised fine-tuning data for the legal RAG LLM.

The goal is to convert QA examples with legal context into instruction-tuning format:

Context + Question → Answer

Input files:
- kaggle_train_qa.csv
- kaggle_val_qa.csv
- kaggle_test_qa.csv

Output files:
- llm_sft_train.jsonl
- llm_sft_val.jsonl
- llm_sft_test.jsonl

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
project_path = "/content/drive/MyDrive/turkish_legal_rag"

processed_path = f"{project_path}/data/processed"
metrics_path = f"{project_path}/outputs/metrics"

print("Processed path:", processed_path)
print("Metrics path:", metrics_path)

Processed path: /content/drive/MyDrive/turkish_legal_rag/data/processed
Metrics path: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics


In [3]:
import os
import re
import json
import pandas as pd
import numpy as np

In [5]:
import os
import glob

print("project_path:", project_path)
print("processed_path:", processed_path)

print("\nProcessed folder exists?", os.path.exists(processed_path))

print("\nFiles in processed folder:")
for file in glob.glob(f"{processed_path}/*"):
    print("-", os.path.basename(file), "|", os.path.getsize(file) / (1024 * 1024), "MB")

project_path: /content/drive/MyDrive/turkish_legal_rag
processed_path: /content/drive/MyDrive/turkish_legal_rag/data/processed

Processed folder exists? True

Files in processed folder:
- val_qa.csv | 0.5993232727050781 MB
- test_qa.csv | 0.4412374496459961 MB
- train_qa.csv | 3.367380142211914 MB
- kaggle_train_qa.csv | 65.28940010070801 MB
- kaggle_val_qa.csv | 13.91445541381836 MB
- kaggle_test_qa.csv | 13.778468132019043 MB
- retrieval_corpus.csv | 1.6800918579101562 MB


In [6]:
def safe_read_csv(path):
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except Exception as e:
        print("Standard read_csv failed:")
        print(type(e).__name__, e)
        print("Trying with python engine and skipping bad lines...")

        return pd.read_csv(
            path,
            encoding="utf-8-sig",
            engine="python",
            on_bad_lines="skip"
        )


kaggle_train_df = safe_read_csv(f"{processed_path}/kaggle_train_qa.csv")
kaggle_val_df = safe_read_csv(f"{processed_path}/kaggle_val_qa.csv")
kaggle_test_df = safe_read_csv(f"{processed_path}/kaggle_test_qa.csv")

print("Train:", kaggle_train_df.shape)
print("Val:", kaggle_val_df.shape)
print("Test:", kaggle_test_df.shape)

display(kaggle_train_df.head())

Train: (9019, 6)
Val: (1933, 6)
Test: (1933, 6)


,soru,cevap,veri türü,kaynak,context,score
0,"Kanunda uygulanabilir bir hüküm yoksa, hâkim n...","Kanunda uygulanabilir bir hüküm yoksa, hâkim ö...",hukuk,Türk Medeni Kanunu,TÜRK MEDENİ KANUNU\r\nKanun Numarası : 4721\r\...,8
1,"Yargıtay Genel Kurulu, Yargıtay üyeleri arasın...","Yargıtay Genel Kurulu, Yargıtay üyeleri arasın...",hukuk,Türkiye Cumhuriyeti Anayasası,ALTINCI KISIM\n\nGEÇİCİ HÜKÜMLER\nGeçici Madde...,10
2,Tapu siciline tescilden önce bir aynî hakkı ka...,"Bir aynî hakkı tescilden önce kazanan kimse, g...",hukuk,Türk Medeni Kanunu,DÖRDÜNCÜ KİTAP\r\nEŞYA HUKUKU\r\nÜÇÜNCÜ KISIM\...,8
3,Eskimiş bayrakların yok edilmesi için hangi ba...,"İçişleri Bakanlığı, Milli Savunma Bakanlığı, D...",hukuk,Türk Bayrağı Tüzüğü,YEDİNCİ BÖLÜM\r\nTescil ve Müsaade İşlemleri\r...,8
4,Göçmen kaçakçılığı suçunun tüzel kişiler açısı...,Göçmen kaçakçılığı suçunun tüzel kişiler açısı...,hukuk,Türk Ceza Kanunu,BİRİNCİ KISIM\r\nUluslararası Suçlar\r\n\r\nİK...,8


In [7]:
for name, df in {
    "train": kaggle_train_df,
    "val": kaggle_val_df,
    "test": kaggle_test_df
}.items():
    print("=" * 100)
    print(name)
    print("Columns:", df.columns.tolist())
    print("Shape:", df.shape)
    display(df.head(2))

train
Columns: ['soru', 'cevap', 'veri türü', 'kaynak', 'context', 'score']
Shape: (9019, 6)


,soru,cevap,veri türü,kaynak,context,score
0,"Kanunda uygulanabilir bir hüküm yoksa, hâkim n...","Kanunda uygulanabilir bir hüküm yoksa, hâkim ö...",hukuk,Türk Medeni Kanunu,TÜRK MEDENİ KANUNU\r\nKanun Numarası : 4721\r\...,8
1,"Yargıtay Genel Kurulu, Yargıtay üyeleri arasın...","Yargıtay Genel Kurulu, Yargıtay üyeleri arasın...",hukuk,Türkiye Cumhuriyeti Anayasası,ALTINCI KISIM\n\nGEÇİCİ HÜKÜMLER\nGeçici Madde...,10


val
Columns: ['soru', 'cevap', 'veri türü', 'kaynak', 'context', 'score']
Shape: (1933, 6)


,soru,cevap,veri türü,kaynak,context,score
0,Bayrak Kanunu'nda bayrağın hangi özellikleri b...,"Bayrak Kanunu'nda bayrağın rengi, ölçüleri, şe...",hukuk,Türk Bayrağı Tüzüğü,DÖRDÜNCÜ BÖLÜM\r\nBayrağın Konulabileceği ve Ö...,10
1,Savaşta yalan haber yayma suçunun kapsamı nedir?,"Savaşta yalan haber yayma suçunun kapsamı, sav...",hukuk,Türk Ceza Kanunu,DÖRDÜNCÜ KISIM\r\nMillete ve Devlete Karşı Suç...,9


test
Columns: ['soru', 'cevap', 'veri türü', 'kaynak', 'context', 'score']
Shape: (1933, 6)


,soru,cevap,veri türü,kaynak,context,score
0,Suçu ve suçluyu övme suçunun cezasının ne kada...,Suçu ve suçluyu övme suçunun cezasının ne kada...,hukuk,Türk Ceza Kanunu,ÜÇÜNCÜ KISIM\r\nTopluma Karşı Suçlar\r\n\r\nBE...,8
1,"Yayımlatan, bedel ödenmesini isteyebilir mi?","Evet, sözleşmede aksi kararlaştırılmış olmadık...",hukuk,Türk Borçlar Kanunu,İKİNCİ KISIM\r\nÖzel Borç İlişkileri\r\nSEKİZİ...,8


In [8]:
def normalize_score_column(df):
    df = df.copy()

    if "score" not in df.columns and "Score" in df.columns:
        df = df.rename(columns={"Score": "score"})

    return df


kaggle_train_df = normalize_score_column(kaggle_train_df)
kaggle_val_df = normalize_score_column(kaggle_val_df)
kaggle_test_df = normalize_score_column(kaggle_test_df)

print(kaggle_train_df.columns.tolist())

['soru', 'cevap', 'veri türü', 'kaynak', 'context', 'score']


In [9]:
for name, df in {
    "train": kaggle_train_df,
    "val": kaggle_val_df,
    "test": kaggle_test_df
}.items():
    print("=" * 100)
    print(name)
    print("Columns:", df.columns.tolist())
    print("Shape:", df.shape)
    print("Score distribution:")
    display(df["score"].value_counts().sort_index())

train
Columns: ['soru', 'cevap', 'veri türü', 'kaynak', 'context', 'score']
Shape: (9019, 6)
Score distribution:


,count
score,
-1,11
0,4
1,12
2,28
3,26
4,14
5,31
6,109
7,714


val
Columns: ['soru', 'cevap', 'veri türü', 'kaynak', 'context', 'score']
Shape: (1933, 6)
Score distribution:


,count
score,
-1,2
1,3
2,6
3,4
4,3
5,5
6,27
7,173
8,1081


test
Columns: ['soru', 'cevap', 'veri türü', 'kaynak', 'context', 'score']
Shape: (1933, 6)
Score distribution:


,count
score,
-1,3
2,4
3,7
4,2
5,6
6,23
7,142
8,1090
9,537


In [10]:
MIN_SCORE = 8

def clean_text(text):
    text = str(text)
    text = text.replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def prepare_qa_df(df, min_score=8):
    df = df.copy()

    df = df.dropna(subset=["soru", "cevap", "context"]).reset_index(drop=True)

    df["soru"] = df["soru"].apply(clean_text)
    df["cevap"] = df["cevap"].apply(clean_text)
    df["context"] = df["context"].apply(clean_text)
    df["kaynak"] = df["kaynak"].astype(str)

    df = df[
        (df["soru"].str.len() > 5) &
        (df["cevap"].str.len() > 5) &
        (df["context"].str.len() > 20)
    ].copy()

    if "score" in df.columns:
        df = df[df["score"] >= min_score].copy()

    df = df.reset_index(drop=True)

    return df


train_clean_df = prepare_qa_df(kaggle_train_df, MIN_SCORE)
val_clean_df = prepare_qa_df(kaggle_val_df, MIN_SCORE)
test_clean_df = prepare_qa_df(kaggle_test_df, MIN_SCORE)

print("Clean train:", train_clean_df.shape)
print("Clean val:", val_clean_df.shape)
print("Clean test:", test_clean_df.shape)

display(train_clean_df.head())

Clean train: (8070, 6)
Clean val: (1710, 6)
Clean test: (1746, 6)


,soru,cevap,veri türü,kaynak,context,score
0,"Kanunda uygulanabilir bir hüküm yoksa, hâkim n...","Kanunda uygulanabilir bir hüküm yoksa, hâkim ö...",hukuk,Türk Medeni Kanunu,TÜRK MEDENİ KANUNU\n\nKanun Numarası : 4721\n\...,8
1,"Yargıtay Genel Kurulu, Yargıtay üyeleri arasın...","Yargıtay Genel Kurulu, Yargıtay üyeleri arasın...",hukuk,Türkiye Cumhuriyeti Anayasası,ALTINCI KISIM\n\nGEÇİCİ HÜKÜMLER\nGeçici Madde...,10
2,Tapu siciline tescilden önce bir aynî hakkı ka...,"Bir aynî hakkı tescilden önce kazanan kimse, g...",hukuk,Türk Medeni Kanunu,DÖRDÜNCÜ KİTAP\n\nEŞYA HUKUKU\n\nÜÇÜNCÜ KISIM\...,8
3,Eskimiş bayrakların yok edilmesi için hangi ba...,"İçişleri Bakanlığı, Milli Savunma Bakanlığı, D...",hukuk,Türk Bayrağı Tüzüğü,YEDİNCİ BÖLÜM\n\nTescil ve Müsaade İşlemleri\n...,8
4,Göçmen kaçakçılığı suçunun tüzel kişiler açısı...,Göçmen kaçakçılığı suçunun tüzel kişiler açısı...,hukuk,Türk Ceza Kanunu,BİRİNCİ KISIM\n\nUluslararası Suçlar\n\nİKİNCİ...,8


In [11]:
MAX_TRAIN_SAMPLES = 4000
MAX_VAL_SAMPLES = 500
MAX_TEST_SAMPLES = 500

train_sft_df = train_clean_df.sample(
    n=min(MAX_TRAIN_SAMPLES, len(train_clean_df)),
    random_state=42
).reset_index(drop=True)

val_sft_df = val_clean_df.sample(
    n=min(MAX_VAL_SAMPLES, len(val_clean_df)),
    random_state=42
).reset_index(drop=True)

test_sft_df = test_clean_df.sample(
    n=min(MAX_TEST_SAMPLES, len(test_clean_df)),
    random_state=42
).reset_index(drop=True)

print("SFT train:", train_sft_df.shape)
print("SFT val:", val_sft_df.shape)
print("SFT test:", test_sft_df.shape)

SFT train: (4000, 6)
SFT val: (500, 6)
SFT test: (500, 6)


In [12]:
SYSTEM_INSTRUCTION = """Sen Türk hukuk metinleri için çalışan dikkatli bir soru-cevap asistanısın.
Cevabı yalnızca verilen bağlama göre üret.
Bağlamda açıkça bulunmayan bilgileri uydurma.
Cevap kısa, net ve Türkçe olmalı."""


def build_mistral_sft_text(row):
    context = row["context"]
    question = row["soru"]
    answer = row["cevap"]

    user_prompt = f"""Bağlam:
{context}

Soru:
{question}

Cevap:"""

    text = f"""<s>[INST] {SYSTEM_INSTRUCTION}

{user_prompt} [/INST] {answer}</s>"""

    return text


train_sft_df["text"] = train_sft_df.apply(build_mistral_sft_text, axis=1)
val_sft_df["text"] = val_sft_df.apply(build_mistral_sft_text, axis=1)
test_sft_df["text"] = test_sft_df.apply(build_mistral_sft_text, axis=1)

print(train_sft_df["text"].iloc[0][:2000])

<s>[INST] Sen Türk hukuk metinleri için çalışan dikkatli bir soru-cevap asistanısın.
Cevabı yalnızca verilen bağlama göre üret.
Bağlamda açıkça bulunmayan bilgileri uydurma.
Cevap kısa, net ve Türkçe olmalı.

Bağlam:
ÜÇÜNCÜ KISIM

Yaptırımlar

İKİNCİ BÖLÜM

Güvenlik Tedbirleri

 

Belli hakları kullanmaktan yoksun bırakılma

MADDE 53. - (1) Kişi, kasten işlemiş olduğu suçtan dolayı hapis cezasına mahkûmiyetin kanuni sonucu olarak;

a) Sürekli, süreli veya geçici bir kamu görevinin üstlenilmesinden; bu kapsamda, Türkiye Büyük Millet Meclisi üyeliğinden veya Devlet, il, belediye, köy veya bunların denetim ve gözetimi altında bulunan kurum ve kuruluşlarca verilen, atamaya veya seçime tâbi bütün memuriyet ve hizmetlerde istihdam edilmekten,

b) Seçme ve seçilme ehliyetinden ve diğer siyasî hakları kullanmaktan,

c) Velayet hakkından; vesayet veya kayyımlığa ait bir hizmette bulunmaktan,

d) Vakıf, dernek, sendika, şirket, kooperatif ve siyasî parti tüzel kişiliklerinin yöneticisi veya dene

In [13]:
def save_jsonl_text(df, path):
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            record = {
                "text": row["text"],
                "question": row["soru"],
                "answer": row["cevap"],
                "source": row["kaynak"],
                "score": float(row["score"]) if "score" in row and pd.notna(row["score"]) else None
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


train_jsonl_path = f"{processed_path}/llm_sft_train.jsonl"
val_jsonl_path = f"{processed_path}/llm_sft_val.jsonl"
test_jsonl_path = f"{processed_path}/llm_sft_test.jsonl"

save_jsonl_text(train_sft_df, train_jsonl_path)
save_jsonl_text(val_sft_df, val_jsonl_path)
save_jsonl_text(test_sft_df, test_jsonl_path)

print("Saved:")
print(train_jsonl_path)
print(val_jsonl_path)
print(test_jsonl_path)

Saved:
/content/drive/MyDrive/turkish_legal_rag/data/processed/llm_sft_train.jsonl
/content/drive/MyDrive/turkish_legal_rag/data/processed/llm_sft_val.jsonl
/content/drive/MyDrive/turkish_legal_rag/data/processed/llm_sft_test.jsonl


In [14]:
llm_sft_data_summary_df = pd.DataFrame([{
    "min_score": MIN_SCORE,
    "original_train_rows": len(kaggle_train_df),
    "original_val_rows": len(kaggle_val_df),
    "original_test_rows": len(kaggle_test_df),
    "clean_train_rows": len(train_clean_df),
    "clean_val_rows": len(val_clean_df),
    "clean_test_rows": len(test_clean_df),
    "sft_train_rows": len(train_sft_df),
    "sft_val_rows": len(val_sft_df),
    "sft_test_rows": len(test_sft_df),
    "train_jsonl_path": train_jsonl_path,
    "val_jsonl_path": val_jsonl_path,
    "test_jsonl_path": test_jsonl_path
}])

llm_sft_data_summary_df

,min_score,original_train_rows,original_val_rows,original_test_rows,clean_train_rows,clean_val_rows,clean_test_rows,sft_train_rows,sft_val_rows,sft_test_rows,train_jsonl_path,val_jsonl_path,test_jsonl_path
0,8,9019,1933,1933,8070,1710,1746,4000,500,500,/content/drive/MyDrive/turkish_legal_rag/data/...,/content/drive/MyDrive/turkish_legal_rag/data/...,/content/drive/MyDrive/turkish_legal_rag/data/...


In [15]:
llm_sft_data_summary_df.to_csv(
    f"{metrics_path}/llm_sft_data_preparation_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("LLM SFT data summary saved.")

LLM SFT data summary saved.


## Data Preparation Conclusion

This notebook prepared context-aware supervised fine-tuning data for the legal RAG LLM.

Instead of using question-answer-only data, the Kaggle QA files were used because they contain legal context, source, question, answer, and quality score columns.

The prepared SFT format teaches the model to answer legal questions based on the provided legal context. This is more aligned with the RAG pipeline than QA-only fine-tuning.